In [1]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import torch.optim as optim

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", device)

Running on: cpu


In [3]:
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [4]:
full_dataset = datasets.ImageFolder("/kaggle/input/with-attetion")
train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_data, val_data, test_data = random_split(full_dataset, [train_size, val_size, test_size])

# Apply transforms
train_data.dataset.transform = train_transform
val_data.dataset.transform = test_transform
test_data.dataset.transform = test_transform

In [5]:
train_loader = DataLoader(train_data, batch_size=2, shuffle=True)
val_loader = DataLoader(val_data, batch_size=2, shuffle=False)
test_loader = DataLoader(test_data, batch_size=2, shuffle=False)

print("Classes:", full_dataset.classes)
print(f"Train: {train_size} | Val: {val_size} | Test: {test_size}")

Classes: ['advanced_glaucoma', 'early_glaucoma', 'normal_control']
Train: 1080 | Val: 231 | Test: 233


In [6]:
class SelfAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.query = nn.Conv2d(channels, channels // 8, 1)
        self.key = nn.Conv2d(channels, channels // 8, 1)
        self.value = nn.Conv2d(channels, channels, 1)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        B, C, H, W = x.size()
        Q = self.query(x).view(B, -1, H * W)
        K = self.key(x).view(B, -1, H * W)
        A = torch.softmax(Q.transpose(1, 2) @ K, dim=-1)
        V = self.value(x).view(B, C, H * W)
        out = (V @ A.transpose(1, 2)).view(B, C, H, W)
        return self.gamma * out + x

In [7]:
class GlaucomaModel(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.backbone = models.densenet121(weights='IMAGENET1K_V1')
        num_ftrs = self.backbone.classifier.in_features
        
        # Remove classifier and get features only
        self.features = self.backbone.features
        self.attn = SelfAttention(num_ftrs)
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(num_ftrs, num_classes)
        )

    def forward(self, x):
        # Extract 4D features (B, C, H, W)
        x = self.features(x)
        x = nn.functional.relu(x)
        
        # Apply attention (needs 4D input)
        x = self.attn(x)
        
        # Classify
        return self.classifier(x)

model = GlaucomaModel(num_classes=len(full_dataset.classes)).to(device)

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth
100%|██████████| 30.8M/30.8M [00:00<00:00, 156MB/s]


In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Training
EPOCHS = 30
best_val_acc = 0

for epoch in range(EPOCHS):
    # Train
    model.train()
    running_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = torch.argmax(model(imgs), dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    val_acc = correct / total
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "glaucoma_model_best.pth")

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {running_loss:.4f} | Val Acc: {val_acc:.4f} | Best: {best_val_acc:.4f}")


Epoch 1/30 | Loss: 448.6209 | Val Acc: 0.6623 | Best: 0.6623
Epoch 2/30 | Loss: 411.1406 | Val Acc: 0.7576 | Best: 0.7576
Epoch 3/30 | Loss: 362.9150 | Val Acc: 0.7359 | Best: 0.7576
Epoch 4/30 | Loss: 353.3037 | Val Acc: 0.7359 | Best: 0.7576
Epoch 5/30 | Loss: 324.9861 | Val Acc: 0.6580 | Best: 0.7576
Epoch 6/30 | Loss: 306.2797 | Val Acc: 0.6883 | Best: 0.7576
Epoch 7/30 | Loss: 294.2473 | Val Acc: 0.7100 | Best: 0.7576
Epoch 8/30 | Loss: 296.1716 | Val Acc: 0.7229 | Best: 0.7576
Epoch 9/30 | Loss: 252.9227 | Val Acc: 0.7273 | Best: 0.7576
Epoch 10/30 | Loss: 233.7329 | Val Acc: 0.7446 | Best: 0.7576
Epoch 11/30 | Loss: 192.6303 | Val Acc: 0.7273 | Best: 0.7576
Epoch 12/30 | Loss: 322.1967 | Val Acc: 0.7576 | Best: 0.7576
Epoch 13/30 | Loss: 213.7302 | Val Acc: 0.7489 | Best: 0.7576
Epoch 14/30 | Loss: 161.2532 | Val Acc: 0.8009 | Best: 0.8009
Epoch 15/30 | Loss: 195.2027 | Val Acc: 0.7749 | Best: 0.8009
Epoch 16/30 | Loss: 166.2102 | Val Acc: 0.6970 | Best: 0.8009
Epoch 17/30 | Los

In [9]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        preds = torch.argmax(model(imgs), dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print("\nFinal Test Accuracy:", correct / total)

# ===========================


Final Test Accuracy: 0.7682403433476395


In [10]:
torch.save(model.state_dict(), "glaucoma_model.pth")
print("Model saved as glaucoma_model.pth")


Model saved as glaucoma_model.pth
